# Lecture 02 Advanced: LoRA on glossary Q&A
Run the cells in order. Runtime → Change runtime type → **T4 GPU** first. When you finish, **Runtime → Disconnect and delete runtime**.

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime -> Change runtime type -> T4 GPU.")

name, memory.total [MiB]
Tesla T4, 15360 MiB



## 1. Install pinned libraries (about 1-2 minutes)

In [2]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 2. Settings
Change `num_train_epochs`, `lr` or `lora_r` here if you want to try another setting.

In [3]:
from pathlib import Path
import os, json, time, re
import pandas as pd

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,     # 116 examples -> about 15 steps per epoch
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 200,     # glossary answers are long
    "gen": {"do_sample": False},
}
WORK = Path("/content/lora_glossary"); WORK.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", None)
print(json.dumps(CONFIG, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 200,
  "gen": {
    "do_sample": false
  }
}


## 3. Load your training data
If the file is not in `/content`, a file picker opens. Choose `glossary_sft.jsonl`.

In [4]:
from datasets import Dataset

DATA = "/content/glossary_sft.jsonl"
if not os.path.exists(DATA):
    from google.colab import files
    up = files.upload()                      # choose glossary_sft.jsonl
    name = next(iter(up))
    os.replace(name, DATA)

records = [json.loads(l) for l in open(DATA, encoding="utf-8") if l.strip()]
train_ds = Dataset.from_list(records)
print(len(records), "records")                # expect 116
print(records[0]["messages"][0]["content"])   # expect: What is AI engineering?

Saving glossary_sft.jsonl to glossary_sft.jsonl
116 records
What is AI engineering?


## 4. Load the base model and define how it is asked a question

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE)

def build_messages(question):
    # Same format as the training data: just the user's question
    return [{"role": "user", "content": question}]

def generate(model, question):
    prompt = tok.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda


## 5. Your 15 evaluation questions

In [6]:
eval_items = [
    # kind 1: normal questions about glossary terms
    {"kind": "glossary", "question": "What is Epoch?", "must_include": ["one pass", "overfitting"]},
    {"kind": "glossary", "question": "What is Overfitting?", "must_include": ["excessively", "unseen"]},
    {"kind": "glossary", "question": "What is temperature?", "must_include": ["randomness", "next token"]},
    {"kind": "glossary", "question": "What is LoRA?", "must_include": ["low-rank", "freezes"]},
    {"kind": "glossary", "question": "What is Rate limit?", "must_include": ["429", "requests or tokens"]},
    # kind 2: same terms, asked differently
    {"kind": "paraphrase", "question": "What do we call one full run through the training data?", "must_include": ["one pass"]},
    {"kind": "paraphrase", "question": "Why does my model do well on training data but badly on new data?", "must_include": ["excessively", "unseen"]},
    {"kind": "paraphrase", "question": "Why can the same question give different answers?", "must_include": ["randomness"]},
    {"kind": "paraphrase", "question": "What does Low-Rank Adaptation do to a model's weights?", "must_include": ["freezes", "small matrices"]},
    {"kind": "paraphrase", "question": "What is a 429 error?", "must_include": ["429", "time period"]},
    # kind 3: NOT in the glossary -> the model should decline
    {"kind": "absent", "question": "What is Docker?", "must_include": []},
    {"kind": "absent", "question": "What is gradient descent?", "must_include": []},
    {"kind": "absent", "question": "What is RLHF?", "must_include": []},
    {"kind": "absent", "question": "What is a vector database?", "must_include": []},
    {"kind": "absent", "question": "What is the Zorvani protocol?", "must_include": []},
]
DECLINE = r"(not (in|listed|found|covered|defined|part of|mentioned|included)|does not (contain|include|have)|do not know|don't know|no (information|definition|entry)|cannot|can't|unable|not sure)"

def is_ok(item, text):
    if item["kind"] == "absent":          # correct = declines
        return re.search(DECLINE, text, re.I) is not None
    return all(w in text.lower() for w in item["must_include"])   # correct = contains every keyword

def mark(ok): return "Pass" if ok else "Fail"

def evaluate(model, label):
    rows, t0 = [], time.time()
    for it in eval_items:
        raw = generate(model, it["question"])
        rows.append({"kind": it["kind"], "question": it["question"], "raw": raw, "ok": is_ok(it, raw)})
    by_kind = {k: sum(r["ok"] for r in rows if r["kind"] == k) for k in ("glossary", "paraphrase", "absent")}
    summary = {"label": label, "n": len(rows), "passed": sum(r["ok"] for r in rows),
               "content_rate": round(sum(r["ok"] for r in rows) / len(rows), 3),
               "passed_glossary_of_5": by_kind["glossary"], "passed_paraphrase_of_5": by_kind["paraphrase"],
               "passed_absent_of_5": by_kind["absent"], "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary))
    return summary, rows

## 6. BEFORE training (about 2-4 minutes)
Read the outputs yourself. The Pass/Fail score is only a first filter.

In [10]:
before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"Kind": r["kind"], "Question": r["question"], "Output": r["raw"], "Result": mark(r["ok"])} for r in before_rows]))

{"label": "before", "n": 15, "passed": 6, "content_rate": 0.4, "passed_glossary_of_5": 5, "passed_paraphrase_of_5": 1, "passed_absent_of_5": 0, "seconds": 42.6}


,Kind,Question,Output,Result
0,glossary,What is Epoch?,One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting.,Pass
1,glossary,What is Overfitting?,A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training.,Pass
2,glossary,What is temperature?,A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers.,Pass
3,glossary,What is LoRA?,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",Pass
4,glossary,What is Rate limit?,"A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have especially strict limits, and agents reach them easily because they call the LLM multiple times for one question.",Pass
5,paraphrase,What do we call one full run through the training data?,"One pass through the training data. In Pydantic AI, this is called ""one fit"". It is the minimum amount of data a model has to learn a given output. If the model fits the data (chooses answers arbitrarily), its answers will change when more data is added.",Pass
6,paraphrase,Why does my model do well on training data but badly on new data?,This is overfitting. The model memorizes the training data rather than generalizing to unseen inputs. Detect it by measuring performance on validation data not used for training.,Fail
7,paraphrase,Why can the same question give different answers?,"Questions vary in difficulty, tone, and expectations. A teacher's role, the level of support, and the homework level all influence answers. In this course, questions are graded by amount and quality of work, not by correctness. Therefore, it is acceptable to give different answers for the same question.",Fail
8,paraphrase,What does Low-Rank Adaptation do to a model's weights?,"Select the smallest possible matrix to be updated, such as the lowest-rank matrix representing one or two predicted values. This reduces the number of weights that change and, in principle, reduces overfitting. In Pydantic AI, it is done with `model.resize_parameters(rank)`.",Fail
9,paraphrase,What is a 429 error?,An error returned when a user exceeds the API's rate limit. The server temporarily denies requests until further rate limit information is received. It is common when calling tools excessively in one go.,Fail


## 7. LoRA training
The progress table shows the Training Loss. Copy the final loss and the training time into your answer.

In [11]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4,
                gradient_accumulation_steps=2, learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=5, save_strategy="no", fp16=False, max_length=CONFIG["max_seq_len"], packing=False,
                report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
train_seconds = round(time.time() - t0)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
final_loss = losses[-1] if losses else None
print(f"Training time: {train_seconds} seconds | Steps: {trainer.state.global_step} | Epochs: {trainer.state.epoch:.1f} | Final loss: {final_loss}")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
5,3.216277
10,2.553137
15,2.190106
20,1.690605
25,1.448986
30,1.292756
35,0.794347
40,0.760425
45,0.525055
50,0.345411


Training time: 155 seconds | Steps: 75 | Epochs: 5.0 | Final loss: 0.23725056648254395


## 8. AFTER training and comparison

In [12]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label"))

regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("Regressions (passed before, failed after):", regressed)

display(pd.DataFrame([{"Kind": b["kind"], "Question": b["question"], "Before": b["raw"], "B": mark(b["ok"]),
                       "After": a["raw"], "A": mark(a["ok"])} for b, a in zip(before_rows, after_rows)]))

{"label": "after", "n": 15, "passed": 7, "content_rate": 0.467, "passed_glossary_of_5": 5, "passed_paraphrase_of_5": 2, "passed_absent_of_5": 0, "seconds": 45.4}


,n,passed,content_rate,passed_glossary_of_5,passed_paraphrase_of_5,passed_absent_of_5,seconds
label,,,,,,,
before,15,6,0.400,5,1,0,42.6
after,15,7,0.467,5,2,0,45.4


Regressions (passed before, failed after): []


,Kind,Question,Before,B,After,A
0,glossary,What is Epoch?,One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting.,Pass,One pass through the entire training dataset is one epoch. Too many epochs can cause overfitting.,Pass
1,glossary,What is Overfitting?,A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training.,Pass,A state in which the model fits training data excessively and performs worse on unseen inputs. Detect it by measuring performance on validation data not used for training.,Pass
2,glossary,What is temperature?,A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers.,Pass,A setting that adjusts randomness when choosing the next token. Lower values make outputs more stable; higher values make them more varied. This is one reason the same question can yield different answers.,Pass
3,glossary,What is LoRA?,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",Pass,"A PEFT method that freezes pretrained weights, represents weight updates as the product of two small matrices (low-rank matrices), and trains only those small matrices. It trains few parameters and saves the results (adapters) in small files. The Lecture 02 exercises cover LoRA.",Pass
4,glossary,What is Rate limit?,"A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have especially strict limits, and agents reach them easily because they call the LLM multiple times for one question.",Pass,"A limit on requests or tokens within a given time period. Exceeding it returns errors such as HTTP 429. Free tiers have especially strict limits, and agents reach them easily because they call the LLM multiple times for one question.",Pass
5,paraphrase,What do we call one full run through the training data?,"One pass through the training data. In Pydantic AI, this is called ""one fit"". It is the minimum amount of data a model has to learn a given output. If the model fits the data (chooses answers arbitrarily), its answers will change when more data is added.",Pass,"One pass through the entire training dataset is one ""training step"". In Pydantic AI, calling `model.train_on_texts(sequences, is_end_turn=True)` one time for one question is one step.",Pass
6,paraphrase,Why does my model do well on training data but badly on new data?,This is overfitting. The model memorizes the training data rather than generalizing to unseen inputs. Detect it by measuring performance on validation data not used for training.,Fail,"This is called overfitting. The model starts fitting training examples and learns noise and quirks specific to that data set. When presented with unseen inputs, it tends to guess incorrectly. To prevent this, add regularization (for example, by adding a small learning rate), evaluate performance on validation data rather than training data, and choose models and datasets suited to the task.",Fail
7,paraphrase,Why can the same question give different answers?,"Questions vary in difficulty, tone, and expectations. A teacher's role, the level of support, and the homework level all influence answers. In this course, questions are graded by amount and quality of work, not by correctness. Therefore, it is acceptable to give different answers for the same question.",Fail,"Inference adds randomness to agent choices, making results slightly different each time. In lecture 03, AI chooses a random answer for each question. In homework, the teacher checks answers exactly as the AI gives them

## 9. Save result.json and download

In [13]:
knowledge_rows = [{"question": b["question"], "before": b["raw"], "before_declined": b["ok"],
                   "after": a["raw"], "after_declined": a["ok"]}
                  for b, a in zip(before_rows, after_rows) if b["kind"] == "absent"]
result = {
    "config": CONFIG,
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (no quantization) on glossary Q&A made by make_sft_dataset.py",
    "n_train_examples": len(records),
    "train_seconds": train_seconds, "final_loss": final_loss, "steps": trainer.state.global_step,
    "before": before_summary, "after": after_summary,
    "regressed": regressed, "knowledge": knowledge_rows,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))

from google.colab import files
for name in ["result.json", "eval_before.json", "eval_after.json"]:
    files.download(str(WORK / name))

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 200,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA (no quantization) on glossary Q&A made by make_sft_dataset.py",
  "n_train_examples": 116,
  "train_seconds": 155,
  "final_loss": 0.23725056648254395,
  "steps": 75,
  "before": {
    "label": "before",
    "n": 15,
    "passed": 6,
    "content_rate": 0.4,
    "passed_glossary_of_5": 5,
    "passed_paraphrase_of_5": 1,
    "passed_absent_of_5": 0,
    "seconds": 42.6
  },
  "after": {
    "label": "after",
    "n": 15,
    "passed": 7,
    "content_rate": 0.467,
    "passed_glossary_of_5": 5,
    "passed_paraphras

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>